# مدل Physics-Informed برای پیش‌بینی IFC و نوار فونون کامل — دیتاست MAX Phase (v2)

**نسخه‌ی اصلاح‌شده — بر اساس نتایج اجرای اول**

### مشکل نسخه‌ی قبلی (v1)
نتایج اجرای اول:

| متریک | مقدار |
|---|---|
| Test IFC MSE | 0.0223 (خوب) |
| Test ASR Loss | 1.860 (بالا — قید فیزیکی نقض شده) |
| Test Symmetry Loss | 0.076 |
| **MAE فرکانس (قابل‌مقایسه با Notebook 11)** | **17.14** ❌ (در مقابل baseline 0.429) |

با وجود اینکه خود IFC با دقت خوبی پیش‌بینی شده بود (MSE پایین)، فرکانس استخراج‌شده از آن کاملاً نادرست بود.

### علت ریشه‌ای
فرکانس از رابطه‌ی `ω = sqrt(eigenvalue)` به‌دست می‌آید — این یک رابطه‌ی **غیرخطی و خیلی حساس** است.
نسخه‌ی قبلی هیچ **ضریب تبدیل واحد فیزیکی** بین `eV/Å²/amu` (واحد طبیعی IFC و جرم) و `THz` (واحد فرکانس) اعمال نمی‌کرد.
بدون این ضریب، eigenvalueها در مقیاس درستی نیستند و فرکانس نهایی چند مرتبه بزرگ‌تر از مقدار واقعی می‌شود.

### اصلاحات این نسخه (v2)
1. ✅ **ضریب تبدیل فیزیکی صحیح** اضافه شد: `eV/Å²/amu → THz²` (همان ثابت استفاده‌شده در Phonopy: `15.633302`)
2. ✅ **افزایش وزن قیدهای فیزیکی**: `λ_ASR` و `λ_symmetry` به‌طور قابل‌توجهی افزایش یافتند تا مدل واقعاً این قیدها را یاد بگیرد (ASR Loss بالا در v1 نشان می‌داد قید رعایت نمی‌شد)
3. ✅ **Re-tuning learning rate و scheduler** برای تناسب با Loss جدید
4. ✅ حفظ اصلاح متیسا: `combined_dim = 8 * hidden` (به‌جای `6 * hidden`)

**Feature Set:** `5_geometry_radius_volume` (برنده‌ی Notebook ۱۱)


## 1. نصب و ایمپورت

In [ ]:
!pip install -q torch_geometric pymatgen torch-cluster torch-sparse torch-scatter torch-spline-conv -t /kaggle/working/custom_lib
import sys
sys.path.append('/kaggle/working/custom_lib')
print("✅ نصب شد")

In [ ]:
import os
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import Set2Set, global_mean_pool, global_max_pool
from torch_geometric.utils import softmax

from scipy.spatial.distance import cdist
from sklearn.metrics import mean_absolute_error
from tqdm.notebook import tqdm

device = torch.device('cpu')  # مطابق آزمایش قبلی
print(f"Device: {device}")

## 2. مسیرهای داده

In [ ]:
FC_DIR       = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
POSCAR_DIR   = '/kaggle/input/datasets/metisa81/pgcnndata/all_poscar/all_poscar'
BANDS_DIR    = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'
ELASTIC_FILE = '/kaggle/input/datasets/metisa81/features-dataset/mechanical_data_fixed.json'
FEATURE_CSV  = '/kaggle/input/datasets/metisa81/feature-dataset-split/5_geometry_radius_volume.csv'

for name, path in [('FC_DIR', FC_DIR), ('POSCAR_DIR', POSCAR_DIR),
                    ('BANDS_DIR', BANDS_DIR), ('ELASTIC_FILE', ELASTIC_FILE),
                    ('FEATURE_CSV', FEATURE_CSV)]:
    exists = '✅' if os.path.exists(path) else '❌ یافت نشد'
    print(f"{exists}  {name}  →  {path}")

## 3. پارسرهای داده

In [ ]:
def parse_poscar(path: str) -> dict:
    with open(path, 'r') as f:
        lines = f.readlines()

    scaling = float(lines[1].strip())
    lattice = np.array(
        [list(map(float, lines[i].split())) for i in range(2, 5)]
    ) * scaling

    line5 = lines[5].strip().split()
    line6 = lines[6].strip().split()

    if line5[0].isalpha():
        elements   = line5
        counts     = list(map(int, line6))
        coord_line = 7
    else:
        counts     = list(map(int, line5))
        elements   = [f'X{i}' for i in range(len(counts))]
        coord_line = 6

    coord_type = lines[coord_line].strip()[0].upper()
    n_atoms    = sum(counts)

    positions = np.array([
        list(map(float, lines[i].split()[:3]))
        for i in range(coord_line + 1, coord_line + 1 + n_atoms)
    ])

    if coord_type == 'D':
        positions = positions @ lattice

    atom_elements = []
    for elem, cnt in zip(elements, counts):
        atom_elements.extend([elem] * cnt)

    return {
        'lattice':       lattice.astype(np.float32),
        'positions':     positions.astype(np.float32),
        'elements':      elements,
        'counts':        counts,
        'atom_elements': atom_elements,
        'n_atoms':       n_atoms,
        'volume':        float(np.abs(np.linalg.det(lattice))),
    }


def parse_force_constants(fc_path: str):
    with open(fc_path, 'r') as f:
        lines = [l.strip() for l in f if l.strip()]

    n_atoms = int(lines[0].split()[0])
    ifcs    = np.zeros((n_atoms, n_atoms, 3, 3), dtype=np.float64)

    idx = 1
    while idx < len(lines):
        parts = lines[idx].split()
        if len(parts) == 2:
            try:
                i, j = int(parts[0]) - 1, int(parts[1]) - 1
                idx += 1
                for row in range(3):
                    vals = list(map(float, lines[idx].split()))
                    ifcs[i, j, row, :] = vals[:3]
                    idx += 1
            except (ValueError, IndexError):
                idx += 1
        else:
            idx += 1

    return n_atoms, ifcs

print("✅ پارسرها آماده شدند")

## 4. ساخت دیتاست خام

In [ ]:
fc_dict     = {Path(f).stem: f for f in Path(FC_DIR).glob('*.fc')}
poscar_dict = {Path(f).stem: f for f in Path(POSCAR_DIR).glob('*.psc')}

band_dict = {}
for f in tqdm(list(Path(BANDS_DIR).glob('*.yaml')), desc="بارگذاری YAML بندها"):
    with open(f, 'r') as file:
        data = yaml.safe_load(file)
        if 'phonon' in data:
            freqs = []
            for point in data['phonon']:
                band_freqs = [b['frequency'] for b in point.get('band', [])]
                if band_freqs:
                    freqs.append(band_freqs)
            if freqs:
                band_dict[f.stem] = np.array(freqs, dtype=np.float32)

common = sorted(set(fc_dict) & set(poscar_dict) & set(band_dict))
print(f"تعداد مواد مشترک: {len(common)}")

shapes = [band_dict[f].shape for f in common]
target_shape = Counter(shapes).most_common(1)[0][0]
print(f"شکل استاندارد نوار فونون (n_qpoints, n_bands): {target_shape}")
N_QPOINTS, N_BANDS = target_shape

with open(ELASTIC_FILE) as f:
    elastic_json = json.load(f)
ELASTIC_KEYS = ['C11','C12','C13','C33','C44','C66',
                'B_Hill','G_Hill','E_Hill','Poisson_Hill',
                'Pugh_ratio','Debye_temperature']
elastic_data = {}
for entry in elastic_json:
    mat = entry.get('material', '')
    vals = [float(entry.get(k, 0) or 0) for k in ELASTIC_KEYS]
    elastic_data[mat] = np.array(vals, dtype=np.float32)

raw_samples = []
failed = []
MAX_ATOMS_FOR_IFC = 12

for formula in tqdm(common, desc="ساخت دیتاست خام"):
    try:
        band_arr = band_dict[formula]
        if band_arr.shape != target_shape:
            continue
        poscar = parse_poscar(str(poscar_dict[formula]))
        n_atoms, ifcs = parse_force_constants(str(fc_dict[formula]))
        if n_atoms != poscar['n_atoms']:
            continue
        if n_atoms > MAX_ATOMS_FOR_IFC:
            continue

        raw_samples.append({
            'formula':           formula,
            'n_atoms':           n_atoms,
            'lattice':           poscar['lattice'],
            'positions':         poscar['positions'],
            'atom_elements':     poscar['atom_elements'],
            'force_constants':   ifcs.astype(np.float32),
            'elastic_constants': elastic_data.get(formula, np.zeros(len(ELASTIC_KEYS), np.float32)),
            'y_phonon':          band_arr.astype(np.float32),
        })
    except Exception as e:
        failed.append((formula, str(e)))

print(f"\n✅ موفق: {len(raw_samples)}  |  ❌ ناموفق/رد شده: {len(failed)}")

## 5. تقسیم Train / Val / Test (seed=42)

In [ ]:
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
idx = rng.permutation(len(raw_samples))

n_total = len(raw_samples)
n_tr = int(0.8 * n_total)
n_va = int(0.1 * n_total)

train_idx = idx[:n_tr]
val_idx   = idx[n_tr:n_tr+n_va]
test_idx  = idx[n_tr+n_va:]

print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

## 6. ساخت گراف (Bond + Atom)

In [ ]:
CUTOFF = 8.0

df_raw = pd.read_csv(FEATURE_CSV)
symbol_col = next((c for c in df_raw.columns if c.lower() in ['symbol', 'element', 'atom']), df_raw.columns[0])
feature_cols = [c for c in df_raw.columns if c not in [symbol_col, 'atomic_number']]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_raw[c])]
df_features = df_raw[feature_cols].copy()
df_features.index = df_raw[symbol_col]
N_ATOM_FEATURES = len(feature_cols)
print(f"فیچرهای اتمی: {N_ATOM_FEATURES} → {feature_cols}")


def structure_to_bond_graph(positions, n_atoms_sample):
    n = len(positions)
    distances = cdist(positions, positions)
    bonds = [(i, j) for i in range(n) for j in range(i+1, n) if distances[i, j] <= CUTOFF]

    if len(bonds) == 0:
        x = torch.zeros((1, 6), dtype=torch.float)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 1), dtype=torch.float)
        u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u, n_atoms=torch.tensor([n_atoms_sample]))

    node_features = []
    for i, j in bonds:
        d = distances[i, j]
        coord_i = np.sum(distances[i] <= CUTOFF) - 1
        coord_j = np.sum(distances[j] <= CUTOFF) - 1
        node_features.append([d, coord_i, coord_j, (coord_i + coord_j) / 2, 0.0, 0.0])
    x = torch.tensor(node_features, dtype=torch.float)

    edge_index, edge_attr = [], []
    for idx1, (i1, j1) in enumerate(bonds):
        for idx2, (i2, j2) in enumerate(bonds[idx1+1:]):
            idx2 += idx1 + 1
            shared = set([i1, j1]) & set([i2, j2])
            if len(shared) == 1:
                s = shared.pop()
                a = positions[i1 if i1 != s else j1] - positions[s]
                b = positions[i2 if i2 != s else j2] - positions[s]
                na, nb = np.linalg.norm(a), np.linalg.norm(b)
                if na > 0 and nb > 0:
                    cos_angle = np.clip(np.dot(a, b) / (na * nb), -1.0, 1.0)
                    angle = np.arccos(cos_angle)
                    edge_index.extend([[idx1, idx2], [idx2, idx1]])
                    edge_attr.extend([[angle], [angle]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous() if edge_index else torch.zeros((2, 0), dtype=torch.long)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float) if edge_attr else torch.zeros((0, 1), dtype=torch.float)
    u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u, n_atoms=torch.tensor([n_atoms_sample]))


def structure_to_atom_graph(atom_elements, positions, n_atoms_sample):
    node_features = []
    for elem in atom_elements:
        if elem in df_features.index:
            feats = df_features.loc[elem].values.astype(np.float32)
        else:
            feats = np.zeros(N_ATOM_FEATURES, dtype=np.float32)
        node_features.append(feats)
    x = torch.tensor(np.array(node_features), dtype=torch.float)

    distances = cdist(positions, positions)
    edge_index, edge_attr = [], []
    n = len(positions)
    for i in range(n):
        for j in range(n):
            if i != j and distances[i, j] <= CUTOFF:
                edge_index.append([i, j])
                edge_attr.append([distances[i, j]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    u = torch.tensor([[1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u, n_atoms=torch.tensor([n_atoms_sample]))

print("✅ توابع ساخت گراف آماده شدند")

In [ ]:
bond_graphs, atom_graphs, ifc_targets, phonon_targets = [], [], [], []

for s in tqdm(raw_samples, desc="ساخت گراف‌ها"):
    bond_graphs.append(structure_to_bond_graph(s['positions'], s['n_atoms']))
    atom_graphs.append(structure_to_atom_graph(s['atom_elements'], s['positions'], s['n_atoms']))
    ifc_targets.append(s['force_constants'])
    phonon_targets.append(s['y_phonon'])

print(f"✅ {len(bond_graphs)} نمونه گراف‌سازی شد")

## 7. معماری مدل: Dual Graph GNN → پیش‌بینی IFC کامل

⚠️ **اصلاح متیسا حفظ شده:** `combined_dim = 8 * hidden` (به‌جای `6 * hidden` در نسخه‌ی اول)


In [ ]:
MAX_ATOMS = max(s['n_atoms'] for s in raw_samples)
print(f"حداکثر تعداد اتم در دیتاست: {MAX_ATOMS}")

In [ ]:
class CustomMessagePassing(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.attention_mlp = nn.Sequential(
            nn.Linear(3 * hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.LeakyReLU(0.2)
        )
        self.message_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )

    def forward(self, x, edge_index, edge_attr):
        source_nodes = edge_index[0]
        target_nodes = edge_index[1]
        neighbor_features = x[source_nodes]
        target_features = x[target_nodes]

        attention_input = torch.cat([target_features, neighbor_features, edge_attr], dim=1)
        attention_scores = self.attention_mlp(attention_input)
        attention_weights = softmax(attention_scores, target_nodes, num_nodes=x.size(0))

        messages = neighbor_features * edge_attr
        weighted_messages = messages * attention_weights

        num_nodes = x.size(0)
        aggregated = torch.zeros(num_nodes, self.hidden_dim, device=x.device)
        aggregated.index_add_(0, target_nodes, weighted_messages)

        return self.message_mlp(aggregated)


class DualGraphIFCNet(nn.Module):
    """Dual Graph GNN برای پیش‌بینی ماتریس IFC کامل."""
    def __init__(self, n_bond_features=6, n_atom_features=7, edge_dim=1, max_atoms=12):
        super().__init__()
        self.max_atoms = max_atoms
        hidden = 128 if n_atom_features <= 10 else 96

        self.bond_embedding = nn.Sequential(
            nn.Linear(n_bond_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(0.1))
        self.bond_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.bond_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(5)])
        self.bond_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(5)])
        self.bond_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.atom_embedding = nn.Sequential(
            nn.Linear(n_atom_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(0.1))
        self.atom_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.atom_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(2)])
        self.atom_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(2)])
        self.atom_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.bond_residual_weight = nn.Parameter(torch.tensor(0.3))
        self.atom_residual_weight = nn.Parameter(torch.tensor(0.3))

        self.set2set_pool = Set2Set(hidden, processing_steps=1)
        self.mean_pool = global_mean_pool
        self.max_pool = global_max_pool
        self.global_mlp = nn.Sequential(nn.Linear(3, hidden // 4), nn.SiLU())

        # ⚠️ اصلاح متیسا: 8 * hidden (نه 6 * hidden)
        combined_dim = 8 * hidden + hidden // 4
        ifc_out_dim = max_atoms * max_atoms * 3 * 3

        self.ifc_head = nn.Sequential(
            nn.Linear(combined_dim, 512), nn.LayerNorm(512), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(512, 256), nn.LayerNorm(256), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(256, ifc_out_dim)
        )

    def _encode_graph(self, x, edge_index, edge_attr, embedding, edge_embedding,
                       message_layers, layer_norms, attention, residual_weight):
        x = embedding(x)
        edge_feat = edge_embedding(edge_attr)
        for i, (msg, ln) in enumerate(zip(message_layers, layer_norms)):
            x_res = x
            x = msg(x, edge_index, edge_feat)
            x = ln(x)
            if i > 0:
                x = x + residual_weight * x_res
            x = F.silu(x)
        attn = attention(x)
        return x * attn, x

    def forward(self, bond_data, atom_data):
        bond_w, x_bond = self._encode_graph(
            bond_data.x, bond_data.edge_index, bond_data.edge_attr,
            self.bond_embedding, self.bond_edge_embedding,
            self.bond_message_layers, self.bond_layer_norms,
            self.bond_attention, self.bond_residual_weight)

        bond_set2set = self.set2set_pool(bond_w, bond_data.batch)
        bond_mean = self.mean_pool(x_bond, bond_data.batch)
        bond_max = self.max_pool(x_bond, bond_data.batch)

        atom_w, x_atom = self._encode_graph(
            atom_data.x, atom_data.edge_index, atom_data.edge_attr,
            self.atom_embedding, self.atom_edge_embedding,
            self.atom_message_layers, self.atom_layer_norms,
            self.atom_attention, self.atom_residual_weight)

        atom_set2set = self.set2set_pool(atom_w, atom_data.batch)
        atom_mean = self.mean_pool(x_atom, atom_data.batch)
        atom_max = self.max_pool(x_atom, atom_data.batch)

        global_features = self.global_mlp(bond_data.u)

        # توجه: 8 بخش escape شده: bond(set2set+mean+max) + atom(set2set+mean+max) + 2 پدینگ صفر برای تطبیق با 8*hidden
        # متیسا این بُعد را برای رفع خطای dimension mismatch به 8 تغییر داد؛ این پدینگ آن منطق را حفظ می‌کند.
        combined = torch.cat([
            bond_set2set, bond_mean, bond_max, zero_pad,
            atom_set2set, atom_mean, atom_max, zero_pad,
            global_features
        ], dim=1)

        ifc_flat = self.ifc_head(combined)
        batch_size = ifc_flat.shape[0]
        ifc_pred = ifc_flat.view(batch_size, self.max_atoms, self.max_atoms, 3, 3)
        return ifc_pred

print("✅ معماری DualGraphIFCNet (v2، با combined_dim=8*hidden) تعریف شد")

## 8. توابع فیزیکی: Dynamical Matrix → نوار فونون — **با ضریب تبدیل واحد صحیح**

### ✅ اصلاح کلیدی این نسخه

رابطه‌ی صحیح برای تبدیل eigenvalue ماتریس دینامیکی (واحد طبیعی `eV/Å²/amu`) به فرکانس `THz`:

```
ω² [THz²] = eigenvalue [eV/Å²/amu] × CONVERSION_FACTOR
CONVERSION_FACTOR = 15.633302   (ثابت استاندارد Phonopy)

f [THz] = sign(ω²) × sqrt(|ω²|) / (2π)
```

این ضریب از رابطه‌ی فیزیکی زیر می‌آید:
```
1 eV/Å² / amu = (1.602176634e-19 J / (1e-10 m)²) / (1.66053906660e-27 kg)
              = 9.648533e+27 rad²/s²
              = (9.648533e+27 / (2π×10¹²)²) THz²
              ≈ 15.633302 THz²
```

نسخه‌ی قبلی (v1) این ضریب را نداشت — به همین خاطر فرکانس‌ها در مقیاس کاملاً نادرستی محاسبه می‌شدند.


In [ ]:
# ثابت تبدیل واحد فیزیکی (eV/Å²/amu → THz²) — مطابق Phonopy
EV_ANGSTROM2_AMU_TO_THZ2 = 15.633302


def compute_phonon_frequencies_gamma(ifc, masses, n_atoms_real):
    """
    محاسبه فرکانس فونون (در واحد THz) در نقطه Gamma (q=0) از ماتریس IFC.

    ifc           : (max_atoms, max_atoms, 3, 3) — IFC پیش‌بینی‌شده [eV/Å²]
    masses        : (max_atoms,) — جرم اتمی [amu]
    n_atoms_real  : تعداد اتم واقعی

    Returns
    -------
    frequencies : (n_atoms_real * 3,) فرکانس‌ها به THz (می‌توانند منفی باشند → مودهای ناپایدار)
    """
    n = n_atoms_real
    ifc_real = ifc[:n, :n, :, :]

    mass_factor = torch.sqrt(masses[:n].unsqueeze(1) * masses[:n].unsqueeze(0))
    dynmat = ifc_real / mass_factor.unsqueeze(-1).unsqueeze(-1)

    dynmat_2d = dynmat.permute(0, 2, 1, 3).reshape(3 * n, 3 * n)
    dynmat_2d = 0.5 * (dynmat_2d + dynmat_2d.T)

    eigenvalues = torch.linalg.eigvalsh(dynmat_2d)  # واحد طبیعی: eV/Å²/amu

    # ✅ اعمال ضریب تبدیل واحد قبل از sqrt
    eigenvalues_thz2 = eigenvalues * EV_ANGSTROM2_AMU_TO_THZ2

    omega = torch.sign(eigenvalues_thz2) * torch.sqrt(torch.abs(eigenvalues_thz2) + 1e-8)  # rad equivalent in THz scale
    frequencies_thz = omega / (2 * np.pi)
    return frequencies_thz


ATOMIC_MASSES = {
    'Ti': 47.87, 'V': 50.94, 'Cr': 52.00, 'Zr': 91.22, 'Nb': 92.91,
    'Mo': 95.95, 'Hf': 178.49, 'Ta': 180.95, 'W': 183.84, 'Sc': 44.96,
    'Al': 26.98, 'Si': 28.09, 'Ga': 69.72, 'Ge': 72.63, 'In': 114.82,
    'Sn': 118.71, 'P': 30.97, 'As': 74.92, 'S': 32.07, 'C': 12.01,
    'N': 14.01,
}

def get_masses_tensor(atom_elements, max_atoms):
    masses = torch.ones(max_atoms, dtype=torch.float32) * 50.0
    for i, elem in enumerate(atom_elements):
        masses[i] = ATOMIC_MASSES.get(elem, 50.0)
    return masses

print("✅ توابع محاسبه فونون با ضریب تبدیل واحد صحیح آماده شدند")

## 9. Physics-Informed Losses — **با وزن‌های افزایش‌یافته**

### تغییر کلیدی این نسخه
در v1، `ASR Loss` نهایی روی Test برابر **1.86** بود — یعنی با وزن قبلی (`λ_asr=0.01`)، مدل اصلاً
انگیزه‌ی کافی برای رعایت این قید را نداشت. در این نسخه وزن‌ها به‌طور قابل‌توجهی افزایش یافته‌اند.


In [ ]:
def symmetry_loss(ifc_pred, n_atoms_real):
    total = 0.0
    batch_size = ifc_pred.shape[0]
    for b in range(batch_size):
        n = n_atoms_real[b]
        ifc = ifc_pred[b, :n, :n, :, :]
        ifc_T = ifc.permute(1, 0, 3, 2)
        total = total + torch.mean((ifc - ifc_T) ** 2)
    return total / batch_size


def acoustic_sum_rule_loss(ifc_pred, n_atoms_real):
    total = 0.0
    batch_size = ifc_pred.shape[0]
    for b in range(batch_size):
        n = n_atoms_real[b]
        ifc = ifc_pred[b, :n, :n, :, :]
        asr = torch.sum(ifc, dim=1)
        total = total + torch.mean(asr ** 2)
    return total / batch_size


def smoothness_loss(frequencies_list):
    total = 0.0
    count = 0
    for freqs in frequencies_list:
        if len(freqs) >= 3:
            d2 = freqs[2:] - 2 * freqs[1:-1] + freqs[:-2]
            total = total + torch.mean(d2 ** 2)
            count += 1
    return total / max(count, 1)

print("✅ Physics losses آماده شدند")

In [ ]:
# ⚠️ وزن‌های افزایش‌یافته نسبت به v1 (که λ_asr=0.01, λ_sym=0.01, λ_smooth=0.001 بود)
LAMBDA_SYMMETRY   = 0.5    # قبلاً 0.01  (افزایش 50x)
LAMBDA_ASR        = 1.0    # قبلاً 0.01  (افزایش 100x — چون ASR Loss در v1 بیشترین نقض را داشت)
LAMBDA_SMOOTHNESS = 0.01   # قبلاً 0.001 (افزایش 10x)


def compute_batch_loss(model, bond_data, atom_data, ifc_target_list, elements_list, device):
    ifc_pred = model(bond_data, atom_data)
    batch_size = ifc_pred.shape[0]
    n_atoms_real = torch.tensor([len(e) for e in elements_list], dtype=torch.long)

    mse_total = 0.0
    freq_preds = []
    for b in range(batch_size):
        n = n_atoms_real[b]
        pred_ifc_b = ifc_pred[b, :n, :n, :, :]
        true_ifc_b = torch.tensor(ifc_target_list[b], dtype=torch.float32, device=device)
        mse_total = mse_total + F.mse_loss(pred_ifc_b, true_ifc_b)

        masses = get_masses_tensor(elements_list[b], MAX_ATOMS).to(device)
        freq_pred_b = compute_phonon_frequencies_gamma(ifc_pred[b], masses, n)
        freq_preds.append(freq_pred_b)

    mse_total = mse_total / batch_size

    sym_loss = symmetry_loss(ifc_pred, n_atoms_real)
    asr_loss = acoustic_sum_rule_loss(ifc_pred, n_atoms_real)
    smooth_loss = smoothness_loss(freq_preds)

    total_loss = (mse_total
                  + LAMBDA_SYMMETRY * sym_loss
                  + LAMBDA_ASR * asr_loss
                  + LAMBDA_SMOOTHNESS * smooth_loss)

    return total_loss, mse_total.item(), sym_loss.item(), asr_loss.item(), smooth_loss.item(), freq_preds

print(f"✅ Loss ترکیبی با وزن‌های جدید: λ_sym={LAMBDA_SYMMETRY}, λ_asr={LAMBDA_ASR}, λ_smooth={LAMBDA_SMOOTHNESS}")

## 10. حلقه‌ی آموزش — **با Learning Rate تنظیم‌شده**

چون مقیاس Loss (به‌خاطر وزن‌های فیزیکی بالاتر) تغییر کرده، نرخ یادگیری و scheduler هم retune شده‌اند:
- LR کوچک‌تر (`0.0003` به‌جای `0.0005`) برای پایداری بیشتر با Loss فیزیکی قوی‌تر
- `ReduceLROnPlateau` به‌جای ثابت بودن، تا اگر Loss فیزیکی نوسان داشت، نرخ یادگیری خودکار کم شود


In [ ]:
def make_batches(indices, batch_size, shuffle=False):
    idx_arr = np.array(indices)
    if shuffle:
        idx_arr = np.random.permutation(idx_arr)
    for i in range(0, len(idx_arr), batch_size):
        yield idx_arr[i:i+batch_size]


def run_epoch(model, indices, optimizer=None, batch_size=8, shuffle=False):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_mse, total_sym, total_asr, total_smooth = 0., 0., 0., 0., 0.
    n_batches = 0

    for batch_idx in make_batches(indices, batch_size, shuffle):
        bond_list = [bond_graphs[i] for i in batch_idx]
        atom_list = [atom_graphs[i] for i in batch_idx]
        ifc_list = [ifc_targets[i] for i in batch_idx]
        elem_list = [raw_samples[i]['atom_elements'] for i in batch_idx]

        bond_batch = Batch.from_data_list(bond_list).to(device)
        atom_batch = Batch.from_data_list(atom_list).to(device)

        with torch.set_grad_enabled(is_train):
            if is_train:
                optimizer.zero_grad()
            loss, mse_v, sym_v, asr_v, smooth_v, _ = compute_batch_loss(
                model, bond_batch, atom_batch, ifc_list, elem_list, device)
            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        total_loss += loss.item()
        total_mse += mse_v
        total_sym += sym_v
        total_asr += asr_v
        total_smooth += smooth_v
        n_batches += 1

    n_batches = max(n_batches, 1)
    return (total_loss/n_batches, total_mse/n_batches, total_sym/n_batches,
            total_asr/n_batches, total_smooth/n_batches)

print("✅ توابع run_epoch آماده شدند")

In [ ]:
model = DualGraphIFCNet(n_bond_features=6, n_atom_features=N_ATOM_FEATURES,
                         edge_dim=1, max_atoms=MAX_ATOMS).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=20)

EPOCHS = 500
PATIENCE = 100
BATCH_SIZE_TRAIN = 8

best_val_loss = float('inf')
best_state = None
patience_counter = 0
history = []

for epoch in range(EPOCHS):
    train_loss, train_mse, train_sym, train_asr, train_smooth = run_epoch(
        model, train_idx, optimizer=optimizer, batch_size=BATCH_SIZE_TRAIN, shuffle=True)

    val_loss, val_mse, val_sym, val_asr, val_smooth = run_epoch(
        model, val_idx, optimizer=None, batch_size=BATCH_SIZE_TRAIN, shuffle=False)

    scheduler.step(val_loss)

    history.append({
        'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
        'val_mse': val_mse, 'val_sym': val_sym, 'val_asr': val_asr, 'val_smooth': val_smooth
    })

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        torch.save(best_state, '/kaggle/working/best_physics_informed_model_v2.pt')
    else:
        patience_counter += 1

    if epoch % 10 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:4d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} "
              f"(MSE={val_mse:.4f}, Sym={val_sym:.4f}, ASR={val_asr:.4f}, Smooth={val_smooth:.4f}) "
              f"LR={current_lr:.6f} ⭐ Best: {best_val_loss:.4f}")

    if patience_counter >= PATIENCE:
        print(f"⚠️ Early stop در epoch {epoch}")
        break

model.load_state_dict(best_state)
print(f"\n✅ آموزش کامل شد. بهترین Val Loss: {best_val_loss:.4f}")

## 11. ارزیابی نهایی روی Test و مقایسه با v1 و Notebook ۱۱

In [ ]:
test_loss, test_mse, test_sym, test_asr, test_smooth = run_epoch(
    model, test_idx, optimizer=None, batch_size=BATCH_SIZE_TRAIN, shuffle=False)

print(f"{'='*60}")
print(f"🏁 نتیجه نهایی — مدل Physics-Informed v2 (ضریب تبدیل واحد + وزن‌های اصلاح‌شده)")
print(f"{'='*60}")
print(f"Test Total Loss:    {test_loss:.4f}")
print(f"Test IFC MSE:        {test_mse:.4f}")
print(f"Test Symmetry Loss:  {test_sym:.6f}")
print(f"Test ASR Loss:       {test_asr:.6f}")
print(f"Test Smoothness:     {test_smooth:.6f}")

print(f"\n{'='*60}")
print("📊 مقایسه با نسخه‌های قبلی")
print(f"{'='*60}")
print(f"Notebook 11 — Test MAE (peak فونون، بدون فیزیک):  0.429")
print(f"Notebook 12 v1 — ASR Loss (وزن کم):                1.860")
print(f"Notebook 12 v1 — MAE فرکانس (بدون ضریب تبدیل):    17.136  ❌")
print(f"Notebook 12 v2 — ASR Loss (وزن بالا):              {test_asr:.4f}")
print(f"Notebook 12 v2 — Test IFC MSE:                      {test_mse:.4f}")

In [ ]:
# استخراج MAE روی فرکانس‌های نقطه گاما — حالا با ضریب تبدیل واحد صحیح
model.eval()
all_freq_mae = []

with torch.no_grad():
    for batch_idx in make_batches(test_idx, BATCH_SIZE_TRAIN, shuffle=False):
        bond_list = [bond_graphs[i] for i in batch_idx]
        atom_list = [atom_graphs[i] for i in batch_idx]
        elem_list = [raw_samples[i]['atom_elements'] for i in batch_idx]
        phonon_list = [phonon_targets[i] for i in batch_idx]

        bond_batch = Batch.from_data_list(bond_list).to(device)
        atom_batch = Batch.from_data_list(atom_list).to(device)
        ifc_pred = model(bond_batch, atom_batch)

        for b in range(len(batch_idx)):
            n = len(elem_list[b])
            masses = get_masses_tensor(elem_list[b], MAX_ATOMS).to(device)
            freq_pred = compute_phonon_frequencies_gamma(ifc_pred[b], masses, n)
            true_peak = float(np.max(phonon_list[b]))
            pred_peak = float(torch.max(torch.abs(freq_pred)).item())
            all_freq_mae.append(abs(true_peak - pred_peak))

freq_mae = np.mean(all_freq_mae)
print(f"🔬 MAE فرکانس (با ضریب تبدیل واحد صحیح، THz): {freq_mae:.4f}")
print(f"   Notebook 11 baseline:        0.429")
print(f"   Notebook 12 v1 (بدون تبدیل): 17.136")
print(f"   Notebook 12 v2 (این نسخه):   {freq_mae:.4f}")

if freq_mae < 0.429:
    print("\n🎉 این مدل بهتر از Notebook 11 عمل کرد!")
elif freq_mae < 17.136:
    print("\n📈 بهبود قابل‌توجه نسبت به v1، اما هنوز به سطح Notebook 11 نرسیده — ممکن است نیاز به تنظیم بیشتر λ یا epoch بیشتر باشد.")
else:
    print("\n⚠️ هنوز بهبودی دیده نمی‌شود — مشکل احتمالاً عمیق‌تر است (نیاز به بررسی دقیق‌تر eigenvalue‌ها).")

## 12. جمع‌بندی و مرحله بعدی

### تغییرات این نسخه (v2) نسبت به v1
| تغییر | v1 | v2 |
|---|---|---|
| ضریب تبدیل واحد | ❌ نداشت | ✅ `15.633302` (eV/Å²/amu → THz²) |
| λ_symmetry | 0.01 | 0.5 |
| λ_asr | 0.01 | 1.0 |
| λ_smoothness | 0.001 | 0.01 |
| Learning Rate | 0.0005 (ثابت) | 0.0003 + ReduceLROnPlateau |
| combined_dim | 6×hidden | 8×hidden (اصلاح متیسا) |

### مراحل بعدی
1. اگر MAE هنوز بهبود کافی نداشت، گام بعدی بررسی دقیق توزیع eigenvalueهای منفی (مودهای ناپایدار) است — ممکن است مدل هنوز IFC فیزیکی معتبر یاد نگرفته باشد.
2. می‌توان `λ_asr` را حتی بیشتر افزایش داد (تا 5 یا 10) اگر هنوز ASR Loss بالا باشد.
3. ثبت نتایج نهایی در `08 - نقشه راه مقاله.md` و `10 - روند Notebook های Kaggle.md`.
4. اگر نتیجه رضایت‌بخش بود، گسترش از نقطه‌ی تک Γ به مسیر کامل q در Brillouin Zone.
